# Participant session export analysis

Download **Full research session JSON** from the admin participant dashboard (`/export/sessions`). Set `EXPORT_PATH` below to that file, or set the `RESEARCH_EXPORT_PATH` environment variable before launching Jupyter. The flat participant summary CSV/JSON is a different export and cannot supply the task, response, or telemetry tables used here.

The bundled data is synthetic and uses the current export fields. Older schema 1.x exports remain supported. Optional or empty sections are displayed as empty tables; missing values are not treated as zero. Workflow identity, demo status, and exported usage attribution are available alongside the raw records.

In [ ]:
import os
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import pandas as pd

from research_toolbox import load_export

# Works when Jupyter starts in the repository, toolbox, or notebooks directory.
PROJECT_ROOT = next(
    root for root in [Path.cwd() / 'research-toolbox', Path.cwd(), *Path.cwd().parents]
    if (root / 'examples' / 'full_sessions.synthetic.json').is_file()
)
EXPORT_PATH = Path(os.environ.get(
    'RESEARCH_EXPORT_PATH', PROJECT_ROOT / 'examples' / 'full_sessions.synthetic.json'
)).expanduser()
EXPORT_PATH

## Load and inspect the contract

In [ ]:
data = load_export(EXPORT_PATH)
display(pd.Series(data.metadata, name='value'))
print('Validation warnings:', data.validation_warnings)
display(pd.DataFrame({'table': list(data.tables), 'rows': [len(frame) for frame in data.tables.values()]}))


def show_columns(frame, columns):
    # Optional sections may have no records or omit fields in historical exports.
    display(frame.reindex(columns=columns))

## Session, workflow, and task progress

`is_demo` marks synthetic participants when supplied by the export. Workflow and configuration IDs identify the source workflow and configuration; `plan_id` and `condition_id` refer to the assigned database records. Keep these distinctions when comparing runs.

In [ ]:
overview = data.session_overview()
show_columns(overview, [
    'session_id', 'state', 'group_name', 'workflow_name', 'workflow_id',
    'configuration_id', 'is_demo', 'condition_id', 'session_duration', 'total_tokens'
])
show_columns(data.task_progress(), [
    'session_id', 'task_id', 'task_type', 'status', 'is_complete', 'elapsed_seconds'
])

## Conditions and outcome metrics

In [ ]:
condition_metrics = data.condition_summary(metrics=['session_duration', 'total_tokens', 'essay_word_count'])
display(condition_metrics)
# Condition IDs represent assigned conditions within a plan, not a pooled workflow condition.
plot_data = overview.reindex(columns=['session_id', 'condition_id', 'total_tokens']).copy()
plot_data['total_tokens'] = pd.to_numeric(plot_data['total_tokens'], errors='coerce')
plot_data = plot_data.dropna(subset=['condition_id', 'total_tokens'])
if plot_data.empty:
    print('No assigned-condition token data to plot.')
else:
    ax = plot_data.plot.bar(x='session_id', y='total_tokens', legend=False, title='Tokens by participant session')
    ax.set_ylabel('tokens')
    plt.tight_layout()
    plt.show()

## Questions, surveys, and essays

In [ ]:
display(data.question_score_summary())
display(data.survey_response_summary())
show_columns(data.essay_summary(), ['session_id', 'essay_id', 'word_count', 'calculated_word_count'])

# Current question definitions use title and description; old examples used prompt.
questions = data.questions.reindex(columns=['question_id', 'title', 'description', 'question_type']).copy()
if 'prompt' in data.questions:
    questions['title'] = questions['title'].fillna(data.questions['prompt'])
response_details = data.question_responses.merge(questions, on='question_id', how='left')
show_columns(response_details, [
    'session_id', 'question_id', 'title', 'description', 'is_answered', 'effective_score'
])

## Chats, exported usage, LLM requests, and telemetry

The export's `session_usage` includes its attribution method, message IDs, and unattributed usage. `task_usage` exposes the per-task totals. `chat_usage_summary()` computes counts from the exported message rows. Message-level task IDs take precedence over the chat's task ID.

The timelines default to the first session in your export. Set `SESSION_ID` to another value from the overview to inspect that participant's run.

In [ ]:
display(data.chat_usage_summary())
show_columns(data.session_usage, [
    'session_id', 'attribution', 'prompts', 'responses', 'input_tokens', 'output_tokens',
    'total_tokens', 'unattributed'
])
display(data.task_usage)
show_columns(data.llm_requests, ['session_id', 'llm_request_id', 'timing_mode', 'status', 'timing_parameters'])
SESSION_ID = next(iter(data.sessions['session_id'].dropna()), None)
print('Timeline session:', SESSION_ID)
timeline = data.telemetry_timeline(SESSION_ID)
show_columns(timeline, ['event_time_dt', 'event_type', 'payload_json'])
show_columns(data.session_timeline(SESSION_ID).head(10), ['occurred_at_dt', 'event_kind', 'source_id'])

## Materialize reusable tables

In [ ]:
with TemporaryDirectory() as directory:
    manifest = data.materialize(
        directory, format='parquet',
        tables=['sessions', 'session_workflows', 'session_usage', 'task_usage', 'session_tasks', 'telemetry_events']
    )
    display(pd.DataFrame(manifest['tables']).T[['file', 'rows', 'json_encoded_columns']])

# Use a durable directory instead of TemporaryDirectory when you want to keep the files.